#建库+写入

In [2]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
import os
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import glob


load_dotenv()

embeddings = OpenAIEmbeddings(
    model="BAAI/bge-m3",
    api_key=os.getenv("SILICONFLOW_API_KEY"),
    base_url="https://api.siliconflow.cn/v1"
)

#建库：指定embedding模型+持久化目录
vectorstore = Chroma(
    collection_name="my_notes",  #集合名
    embedding_function=embeddings,
    persist_directory="chroma_db"  #数据存到本地文件夹
)

#写入
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

def load_text(path:str):
    with open(path,encoding="utf-8") as f:
        text = f.read()
    return Document(page_content=text,metadata={"source":path})

files = glob.glob("data/*.md")
all_docs = [load_text(f) for f in files]
chunks = splitter.split_documents(all_docs)

#vectorstore.add_documents(chunks)
print("已写入条数：",vectorstore._collection.count())

已写入条数： 186


#相似度检索

In [3]:
question = "怎么将项目推送到github？"
hits = vectorstore.similarity_search(question,k=3)
print(len(hits))
for d in hits:
    print("来源:",d.metadata.get("source"))
    print("内容",d.page_content[:60])
    print("---")

3
来源: data\rag_notes_1.md
内容 | 域名（可选） | HTTPS + 简历好看 | 阿里云等 | .xyz 约 ¥10/年 |
---
来源: data\第一周实操指南_Day1-9.md
内容 14. 合并两个列表 [1,2,3] 和 [4,5,6] 成 [1,4,2,5,3,6]（用 zip）
15. 递归写阶
---
来源: data\rag_notes_1.md
内容 | 9/6（Day 37） | pytest 基础：给分块、检索、chat 接口写测试 | 测试通过 |
---


#带分数检索  Chroma 返回的是**距离**（越小越相似，不是 0-1 的相似度）。用它能判断"这次检索质量高不高"——分数普遍很低说明很相关，都很大说明没检索到好东西。

In [11]:
hits_with_score = vectorstore.similarity_search_with_score(question,k=3)
for doc,score in hits_with_score:
    print(round(score,4))
doc.page_content[:40]

0.6584
0.6907
0.7074


'| 9/6（Day 37） | pytest 基础：给分块、检索、chat 接口'

#metadata过滤（按来源筛）

In [14]:
hits = vectorstore.similarity_search(
    question,k=3,
    filter={"source":"data\\rag_notes_1.md"} #只在这个文件里找
)
print([d.metadata["source"] for d in hits])

['data\\rag_notes_1.md', 'data\\rag_notes_1.md', 'data\\rag_notes_1.md']


In [13]:
items = vectorstore.get(limit=50)
print(sorted(set(m.get("source") for m in items["metadatas"])))

['data\\rag_notes_1.md']
